In [8]:
from arcgis.gis import GIS
import json

gis = GIS("home")

# ---------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------
old_item_id = "91a7e4ee709646febe991c74541ac2e7" #Audubon Assets Public View (deprecated)
new_item_id = "fc37bb5a33494a8e90aa1278c0aa9188" #Spaces and Places Points AirTable (VIEW) Public

webmap_ids = [
    "d3e39e8a39404da384f0f13ead7c0e93", #replace layer test map
    #"WEBMAP_ID_2",
    #"WEBMAP_ID_3"
]
# ---------------------------------------------------------


def replace_layer_recursive(layers, old_id, new_id):
    """
    Recursively walk through operationalLayers or group layers
    and replace layers whose itemId matches old_id.
    """
    count = 0

    for lyr in layers:
        item_id = lyr.get("itemId")

        # Normal layer replacement
        if item_id == old_id:
            print(f"   🔁 Replacing layer '{lyr.get('title')}' → {new_id}")
            lyr["itemId"] = new_id
            if "url" in lyr:
                del lyr["url"]   # important or AGOL won't refresh properly
            count += 1

        # Recurse into group layers
        if "layers" in lyr:
            count += replace_layer_recursive(lyr["layers"], old_id, new_id)

    return count


# ---------------------------------------------------------
# PROCESS EACH WEB MAP
# ---------------------------------------------------------
for wm_id in webmap_ids:
    print(f"\n📍 Processing web map: {wm_id}")

    webmap_item = gis.content.get(wm_id)
    if not webmap_item:
        print(f"❌ Web map {wm_id} not found.")
        continue

    data = webmap_item.get_data()

    if "operationalLayers" not in data:
        print("⚠️ No operational layers found. Skipping.")
        continue

    # Replace occurrences in this map
    replaced = replace_layer_recursive(data["operationalLayers"], old_item_id, new_item_id)

    if replaced > 0:
        # Save back to AGOL
        webmap_item.update(item_properties={"text": json.dumps(data)})
        print(f"✅ Updated {wm_id}: replaced {replaced} layers.")
    else:
        print(f"⚠️ No layers matched {old_item_id} in {wm_id}.")

print("\n🎉 Done!")



📍 Processing web map: d3e39e8a39404da384f0f13ead7c0e93
   🔁 Replacing layer 'Audubon Assets Public View' → fc37bb5a33494a8e90aa1278c0aa9188
✅ Updated d3e39e8a39404da384f0f13ead7c0e93: replaced 1 layers.

🎉 Done!
